In [ ]:
#| hide
from slopometer.score import *


# slopometer

> Score prose for simplicity and precision against plain-English reference rules

`slopometer` measures prose against a range of reference-prose rules. The rules merge GOV.UK/GDS house style with the discipline of ASD-STE100, including 26 numbered tells, banned-word lists, and a reference register. It also has rules covering a range of best practices for clear technical communication, including paragraph length, referant tracking, and restatement.

`slopometer` is for anyone shipping READMEs, docstrings, PR text, or agent responses. It catches slop mechanically, in milliseconds, with no LLM in the loop. Every finding cites a rule, quotes a span, and carries a weight.


## Install

    pip install slopometer

The first score on a machine downloads spaCy's `en_core_web_md` model (about 40MB, once) into `~/.cache/slopometer` and loads it by path. The model never enters a virtual environment. One copy serves every project. Environment syncs cannot remove it.

## Use

In [ ]:
from slopometer.score import score_text, score_path

The default minimum is 150 scored words. Shorter inputs return `too short to meter` with their word count and no numeric score. Set `min_words=0` to meter short examples:


In [ ]:
score_text("This section describes our approach. It isn't just a linter - it's a comprehensive paradigm for quality.", min_words=0)

density 252.9 (weight 43 on 17 prose words), worst 10
1: [10] notxbuty (tell 16, not-X-but-Y): "isn't just a"
1: [10] splice (tell 1, splices): ' - '
1: [10] banned: 'comprehensive' -> 'complete'
1: [10] banned: 'paradigm'
1: [3] throat_clearing (tell 13, throat-clearing): 'This section describes'

Density is weighted findings per 100 words across paragraphs, headings, and list items. Heading and list markers do not count as words. Code blocks and other content excluded from scoring do not enter the denominator or count toward the minimum.

`score_text`, `score_path`, and `score_many` accept `min_words`, defaulting to 150. A short result has `too_short=True`, an empty findings list, and `None` for `density`, `total`, and `worst`.

`score_path` scores Markdown files or the Markdown cells of an `.ipynb` notebook. Notebook cells are joined with blank lines and scored as one document; code cells, outputs, and raw cells are excluded. File reports carry `lineno|hash|` addresses, prefixed with the cell ID for notebooks. `Result.location(finding)` returns the source location. The command line accepts a file or Markdown on stdin:

    slopometer --path README.md
    slopometer --path nbs/00_core.ipynb
    git log -1 --format=%B | slopometer --min-words 0
    slopometer --path draft.md --threshold 10

`--threshold` returns exit code 1 when a score exceeds the limit. Inputs below `--min-words` print the short-input message and exit successfully without scoring. JSON output marks them with `too_short: true`, includes `min_words`, and uses null scores. Notebook findings include a `location` object with `cell_id`, zero-based `cell_index`, one-based `line`, and `address`. Their `start` and `end` offsets refer to the combined Markdown, not notebook JSON.

The command runs warm through `warmpy`. The first input long enough to score loads the model in a background process. Later calls answer in milliseconds. After thirty idle minutes the process exits.

## What it checks, and what it cannot

The rules live in notebooks that teach each family beside its code. Each rule states its tell, shows a violating example, and shows the plain rewrite. The [lexicon notebook](https://AnswerDotAI.github.io/slopometer/lexicon.html) holds the word and phrase rules. The [syntax notebook](https://AnswerDotAI.github.io/slopometer/syntax.html) builds sentence rules on spaCy's parse. The [para notebook](https://AnswerDotAI.github.io/slopometer/para.html) covers paragraph and document rules and the limits of word-vector heuristics. The [score notebook](https://AnswerDotAI.github.io/slopometer/score.html) assembles the pipeline. A drift test asserts that every `write_docs` tell maps to a rule or to an explicit unscoreable registry. The meter and the style guide cannot drift apart silently.

Vocabulary novelty is not scored. Identifying unexplained terminology requires audience and document context.

A rule ships only when its false-positive rate on clean reference prose is near zero. The meter scores the style guide's own clean passage at exactly 0.0. The score notebook measures the blind spot instead of hiding it: mechanically chopped prose passes every surface rule while staying opaque. Agent review (`check_docs`) and an optional, explicitly invoked [Pangram](https://www.pangram.com) check cover that residue. Vale, write-good, and proselint solve neighboring problems. The [lexicon notebook](https://AnswerDotAI.github.io/slopometer/lexicon.html) records what came from them. It also credits the [GOV.UK words-to-avoid list](https://www.gov.uk/guidance/style-guide/a-to-z-of-gov-uk-style#words-to-avoid) (OGL v3).
